# Tail-only excerpt: can the error window be dropped?

The two-stream experiment (`01_two_stream_gate.ipynb`) suggested that the log tail carries most of the failure signal. Removing the error-window branch from the *trained* two-stream model barely changed test accuracy (0.924 → 0.918), while removing the tail cost much more (0.924 → 0.860).

That knockout didn't retrain anything, though. This notebook asks the follow-up question before changing the app: **does the original MLP, retrained on the log tail alone, do as well as it does on the current excerpt?**

| Variant | Excerpt |
|---|---|
| `current` | First error match (5 before / 15 after) + last 30 lines, merged. This was the app's excerpt when this experiment was run |
| `tail_30` | Last 30 lines only |
| `tail_50` | Last 50 lines only |
| `tail_80` | Last 80 lines only |

Everything else matches the original model: the same rows, splits and labels, the same masking and 4,000-character cap, TF-IDF reduced to 256 dimensions, the same `TrainConfig`, and the same seeds. Only the excerpt changes.

**Decision rule.** Switch the app to tail-only if some tail length is within run-to-run noise of `current` on both validation and test, *and* `build_execution` doesn't lose more than a log or two. That class lost the most when the window was knocked out in the two-stream model.

**To run:** same setup as notebook 01. Start Jupyter from `experiments/`, with `DB_PATH` (or `YNB_DB_PATH`) pointing at the annotated database.

In [ ]:
import copy
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from ynbtriage.db import connect
from ynbtriage.model.data import label_order, load_dataset
from ynbtriage.model.excerpt import normalize
from ynbtriage.model.features import make_featurizer
from ynbtriage.model.metrics import score
from ynbtriage.model.pipeline import fit_baselines, prepare
from ynbtriage.model.train import TrainConfig, train_mlp

DB_PATH = os.environ.get("YNB_DB_PATH")   # or e.g. "../ynobuild-final-annotations.db"
SEEDS = (7400, 7401, 7402, 7403, 7404)
TAIL_LENGTHS = (30, 50, 80)
CFG = TrainConfig()   # original settings: Adam 1e-3, wd 1e-4, dropout 0.2, batch 32, patience 25
print("DB:", DB_PATH)

## 1. Data and excerpts

`load_dataset` gives the same rows, splits and labels as the original model, with the current excerpt in `text`. The raw `log_tail` is fetched again by `build_id` so the tail-only excerpts can be built from it.

In [ ]:
def tail_excerpt(log: str, lines: int, max_chars: int = 4000) -> str:
    # Last `lines` lines, with the same masking and character cap as the current excerpt.
    kept = (log or "").splitlines()[-lines:]
    return normalize("\n".join(kept))[-max_chars:]


df = load_dataset(DB_PATH, excerpt="error_window")   # the original excerpt rule
conn = connect(DB_PATH)
try:
    logs = pd.read_sql_query("SELECT build_id, log_tail FROM builds", conn)
finally:
    conn.close()
df = df.merge(logs, on="build_id", how="left")

df["text_current"] = df["text"]
for n in TAIL_LENGTHS:
    df[f"text_tail_{n}"] = df["log_tail"].map(lambda s, n=n: tail_excerpt(s, n))

df, labels, dropped, parts, y, _ = prepare(df, label_order(DB_PATH))   # REUSED from pipeline.py
VARIANTS = ["current"] + [f"tail_{n}" for n in TAIL_LENGTHS]
texts = {v: {s: d[f"text_{v}"].tolist() for s, d in parts.items()} for v in VARIANTS}

print("labels:", labels)
print({s: len(d) for s, d in parts.items()})

In [ ]:
# How much text does each variant give the model? (characters per excerpt, train split)
pd.DataFrame({v: pd.Series([len(t) for t in texts[v]["train"]]).describe(percentiles=[.5])
              for v in VARIANTS}).loc[["mean", "50%", "max"]].round(0)

## 2. Train the original MLP on each excerpt

Every variant gets its own TF-IDF featurizer, fit on train only, and its own MLP, for each seed. The shallow baselines are refit per variant too, since they would change along with the excerpt in the app.

In [ ]:
runs = []        # one row per (variant, seed, split)
preds = {}       # (variant, seed) -> test predictions, for the paired comparison below
baseline_rows = []

for v in VARIANTS:
    feat = make_featurizer("tfidf", tfidf_dim=256).fit(texts[v]["train"])
    X = {s: feat.transform(t) for s, t in texts[v].items()}
    for seed in SEEDS:
        c = copy.deepcopy(CFG); c.seed = seed
        model, hist = train_mlp(X["train"], y["train"], X["val"], y["val"], len(labels), c)
        for split in ("val", "test"):
            pred = model.predict_proba(torch.tensor(X[split])).numpy().argmax(1)
            s = score(y[split], pred, labels)
            correct = {f"correct_{l}": int(((pred == y[split]) & (y[split] == i)).sum())
                       for i, l in enumerate(labels)}
            runs.append({"variant": v, "seed": seed, "split": split, "epochs": len(hist),
                         "accuracy": s["accuracy"], "macro_f1": s["macro_f1"],
                         **{f"f1_{k}": val for k, val in s["per_class_f1"].items()}, **correct})
            if split == "test":
                preds[(v, seed)] = pred
    b = fit_baselines(texts[v], y, labels)
    for split in ("val", "test"):
        baseline_rows.append({"variant": v, "split": split,
                              "logreg_macro_f1": b["tfidf_logreg"][split]["macro_f1"],
                              "naive_bayes_macro_f1": b["naive_bayes"][split]["macro_f1"]})
    print(f"{v:>8}: done")

runs = pd.DataFrame(runs)
baselines = pd.DataFrame(baseline_rows)

## 3. Results

In [ ]:
# Headline: macro-F1 and accuracy, mean ± SD over seeds
summary = (runs.groupby(["split", "variant"], sort=False)[["accuracy", "macro_f1"]]
           .agg(["mean", "std"]).round(3))
summary

In [ ]:
# Per-class F1 (mean over seeds). Watch build_execution.
f1_cols = [f"f1_{l}" for l in labels]
(runs.groupby(["split", "variant"], sort=False)[f1_cols].mean().round(3)
     .rename(columns=lambda c: c.removeprefix("f1_")))

In [ ]:
# The same thing in logs: how many logs of each class were classified correctly (mean over seeds).
# This is the unit the decision rule uses ("a log or two").
cc = [f"correct_{l}" for l in labels]
counts = (runs.groupby(["split", "variant"], sort=False)[cc].mean().round(1)
              .rename(columns=lambda c: c.removeprefix("correct_")))
class_n = {s: pd.Series(y[s]).map(dict(enumerate(labels))).value_counts().reindex(labels)
           for s in ("val", "test")}
print("logs per class:", {s: class_n[s].to_dict() for s in class_n})
counts

In [ ]:
# Shallow baselines on each excerpt (one fit each; these are deterministic)
baselines.set_index(["split", "variant"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, split in zip(axes, ("val", "test")):
    d = runs[runs["split"] == split]
    for i, v in enumerate(VARIANTS):
        vals = d.loc[d["variant"] == v, "macro_f1"]
        ax.plot([i] * len(vals), vals, "o", color="#2a78d6" if v != "current" else "#666",
                alpha=0.6, ms=6)
        ax.plot([i - 0.2, i + 0.2], [vals.mean()] * 2, color="#222", lw=2)
    ax.set_xticks(range(len(VARIANTS)), VARIANTS)
    ax.set_title(f"Macro-F1, {split} (dots = seeds, bar = mean)", loc="left", fontsize=11)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
fig.tight_layout()

## 4. Paired comparison against the current excerpt

Each tail variant is compared to `current` **seed by seed**, since both were trained with the same seed. That removes part of the run-to-run noise.

The bootstrap resamples the test logs to show how much the difference could move on a different set of test logs. That uncertainty is usually larger than the seed-to-seed spread.

In [ ]:
rng = np.random.default_rng(0)
yt = y["test"]
rows = []
for v in VARIANTS[1:]:
    seed_diffs = [(preds[(v, s)] == yt).mean() - (preds[("current", s)] == yt).mean() for s in SEEDS]
    # bootstrap the test logs, using predictions averaged over seeds (share of seeds that got each log right)
    ok_v = np.mean([preds[(v, s)] == yt for s in SEEDS], axis=0)
    ok_c = np.mean([preds[("current", s)] == yt for s in SEEDS], axis=0)
    boot = []
    for _ in range(2000):
        idx = rng.integers(0, len(yt), len(yt))
        boot.append(ok_v[idx].mean() - ok_c[idx].mean())
    lo, hi = np.percentile(boot, [2.5, 97.5])
    rows.append({"variant": v,
                 "test acc diff vs current (mean over seeds)": np.mean(seed_diffs),
                 "sd over seeds": np.std(seed_diffs, ddof=1),
                 "bootstrap 95% interval": f"[{lo:+.3f}, {hi:+.3f}]",
                 "diff in logs (test)": np.mean(seed_diffs) * len(yt)})
pd.DataFrame(rows).round(3)

## 5. Decision

In [ ]:
# Applies the decision rule from the top of the notebook to the numbers above.
# Noise threshold: 2 SD of the current excerpt's run-to-run macro-F1 spread, or roughly one log
# (1 / number of logs in the split), whichever is larger.
m = runs.groupby(["split", "variant"])["macro_f1"].agg(["mean", "std"])
cnt = runs.groupby(["split", "variant"])["correct_build_execution"].mean()
for v in VARIANTS[1:]:
    ok = True
    notes = []
    for split in ("val", "test"):
        cur_mean, cur_sd = m.loc[(split, "current")]
        thresh = max(2 * cur_sd, 1 / len(y[split]))
        gap = m.loc[(split, v), "mean"] - cur_mean
        within = gap >= -thresh
        ok &= within
        notes.append(f"{split}: macro-F1 {gap:+.3f} (allowed ≥ {-thresh:.3f}) {'✓' if within else '✗'}")
        be_drop = cnt.loc[(split, "current")] - cnt.loc[(split, v)]
        be_ok = be_drop <= 2
        ok &= be_ok
        notes.append(f"{split}: build_execution correct {-be_drop:+.1f} logs (allowed ≥ -2) {'✓' if be_ok else '✗'}")
    print(f"{v}: {'PASSES' if ok else 'does not pass'}")
    for n in notes:
        print("   ", n)

**Reading the results:**

- If one or more tail lengths **pass**, prefer the shortest one that passes. It's the simplest rule and gives the model the least boilerplate to sift through. That's the excerpt to carry into Stage 2 (the app changes).
- If **none pass**, keep the current excerpt. In that case the error window helps a retrained model even though the two-stream model leaned away from it. That's worth a line in `docs/experiments.md`, since it refines the knockout result.
- Check the bootstrap intervals either way. If an interval comfortably includes 0, the variants are indistinguishable on this test set, whichever way the means point.